# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, to_date

In [0]:
RENAME_MAP = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'create_date'
}

# Lendo dados da Tabela Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Transformação de Dados


In [0]:
# Usar trim em string
# Normalizar dados em marital_status e gndr
# Renomear colunas


for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()
    

# Normalização



In [0]:
df = (
    df.withColumn(
        'cst_marital_status', 
        F.when(F.upper(F.col("cst_marital_status")) == 'S', 'Single')
        .when(F.upper(F.col("cst_marital_status"))  == 'M', 'Maried')
        .otherwise('n/a')
    )
    .withColumn(
        'cst_gndr',
        F.when(F.upper(F.col("cst_gndr")) == 'M', 'Male')
        .when(F.upper(F.col("cst_gndr")) == 'F', 'Female')
        .otherwise('n/a')
    )
)

df.display()

# Renomeando colunas

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.display()


# Escrevendo na Tabela Silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.customer_info")

In [0]:
%sql
SELECT * FROM workspace.silver.customer_info